# 面试问题：怎样搭建可靠的 LLM/Agent 评测体系，而不是“人工看几个答案”？

**一句话回答**：从真实任务与失败日志构造带版本、来源、slice 和风险等级的黄金集；同时使用确定性 grader、规则/模型 grader 与人工复核；报告任务成功、质量、安全、延迟、成本及置信区间；发布采用 paired 对比和分 slice 门禁，线上再用 shadow、抽样审计与漂移补集闭环。

本 Notebook 从零实现黄金集合同、文本指标、轨迹检查、paired bootstrap、slice 门禁和数据污染审计。

In [ ]:
from dataclasses import dataclass
from collections import Counter, defaultdict
import hashlib, json, math, re
import numpy as np

SEED104=10401; rng104=np.random.default_rng(SEED104)
assert SEED104==10401
assert re.findall(r"\w+","答案 42")==["答案","42"]
assert hashlib.sha256(b"gold-v1").hexdigest()!=hashlib.sha256(b"gold-v2").hexdigest()

## 1. Eval case 是版本化测试合同

每条包含稳定 ID、输入、期望/证据、允许工具、slice、风险和来源时间。黄金答案不一定是唯一字符串，可写成结构化约束或 grader。生产日志入集前要脱敏、去重和人工确认，避免把模型自己的错误输出当真值。

In [ ]:
@dataclass(frozen=True)
class EvalCase104:
    case_id:str; query:str; expected:str; slice:str; risk:str="normal"; source_time:int=0
    def __post_init__(self):
        if not self.case_id or not self.query or not self.slice or self.risk not in {"normal","high"}: raise ValueError("case_contract")
cases104=[EvalCase104("c1","2+2 等于多少","4","math"),EvalCase104("c2","退款期多久","7天","policy","high"),EvalCase104("c3","首都是哪里","北京","fact"),EvalCase104("c4","3+5","8","math")]
assert len({c.case_id for c in cases104})==len(cases104)
assert Counter(c.slice for c in cases104)=={"math":2,"policy":1,"fact":1}
try: EvalCase104("","q","a","s"); raise AssertionError("bad case accepted")
except ValueError as e: assert str(e)=="case_contract"

## 2. 数据集要覆盖能力、分布与风险，而非只追平均分

题目来源可分人工核心集、线上回放、对抗集和新鲜集；slice 至少覆盖语言、长度、租户、工具、拒答与高风险场景。固定 regression set 用于版本比较，滚动 challenge set 防止团队对测试集过拟合。

In [ ]:
def stratified_sample104(rows,n_per_slice,seed):
    groups=defaultdict(list)
    for r in rows: groups[r.slice].append(r)
    out=[]; rg=np.random.default_rng(seed)
    for key in sorted(groups):
        idx=rg.choice(len(groups[key]),size=min(n_per_slice,len(groups[key])),replace=False); out.extend(groups[key][int(i)] for i in idx)
    return out
sample_a104=stratified_sample104(cases104,1,7); sample_b104=stratified_sample104(cases104,1,7)
assert [c.case_id for c in sample_a104]==[c.case_id for c in sample_b104]
assert {c.slice for c in sample_a104}=={"math","policy","fact"}
assert len(sample_a104)==3

## 3. 优先使用可解释的确定性 grader

结构化任务先校验 JSON/schema、数值容差、引用 ID、工具轨迹或 exact match。开放文本可补 token F1，但它不理解事实；只有规则无法覆盖的主观维度才交给模型或人工。grader 也要单测，否则评测系统会制造假回归。

In [ ]:
def normalize104(s): return re.findall(r"[\w]+",s.lower(),flags=re.UNICODE)
def token_f1104(pred,gold):
    p,g=Counter(normalize104(pred)),Counter(normalize104(gold)); common=sum((p&g).values())
    if not p or not g: return float(p==g)
    precision,recall=common/sum(p.values()),common/sum(g.values()); return 0. if common==0 else 2*precision*recall/(precision+recall)
assert token_f1104("退款期 7 天","退款期 7 天")==1
assert math.isclose(token_f1104("退款期 是 7 天","退款期 7 天"),6/7)
assert token_f1104("完全错误","退款期")==0

## 4. Agent 要评轨迹，而不只看最终文字

最终答案正确也可能调用了未授权工具、泄露数据或浪费 30 步。轨迹 grader 检查 allowed tools、调用参数、预算、是否有写操作审批、最终证据是否来自 observation。结果同时保留 outcome 与 process 指标。

In [ ]:
def grade_trace104(trace,allowed,max_steps=5):
    violations=[]
    if len(trace)>max_steps: violations.append("step_budget")
    for e in trace:
        if e["tool"] not in allowed: violations.append("unauthorized_tool")
        if e.get("write") and not e.get("approved"): violations.append("unapproved_write")
    return {"pass":not violations,"violations":violations,"steps":len(trace)}
good_trace104=[{"tool":"search","write":False},{"tool":"calculator","write":False}]; bad_trace104=[{"tool":"email","write":True,"approved":False}]
assert grade_trace104(good_trace104,{"search","calculator"})["pass"]
assert not grade_trace104(bad_trace104,{"search"})["pass"]
assert set(grade_trace104(bad_trace104,{"search"})["violations"])=={"unauthorized_tool","unapproved_write"}

## 5. 版本比较要 paired，而不是比较两个孤立均值

同一 case 上 challenger 与 baseline 的差值能消除题目难度噪声。下面对 paired improvement 做 bootstrap 置信区间；若区间跨 0，就不能仅凭点估计宣称提升。采样单位应与独立单位一致，例如按用户或会话 bootstrap。

In [ ]:
base104=np.array([1,0,1,0,1,0,0,1,0,1],float); new104=np.array([1,1,1,0,1,1,0,1,1,1],float); delta104=new104-base104
def paired_bootstrap104(delta,reps=3000,seed=0):
    rg=np.random.default_rng(seed); means=np.array([delta[rg.integers(0,len(delta),len(delta))].mean() for _ in range(reps)]); return float(delta.mean()),tuple(np.quantile(means,[.025,.975]))
gain104,ci104=paired_bootstrap104(delta104,seed=104)
assert math.isclose(gain104,.3)
assert ci104[0]>=0 and ci104[1]<=.7
assert paired_bootstrap104(delta104,seed=104)==(gain104,ci104)

## 6. 总分上涨不能掩盖关键 slice 退化

对高风险、语言、长上下文、权限边界分别设 non-regression gate。小 slice 的区间会很宽，应报告样本数并人工复核，而不是静默删除。宏平均体现 slice 公平，微平均体现流量加权，两者都应保留。

In [ ]:
rows104=[("normal",1,1),("normal",0,1),("normal",1,1),("high",1,1),("high",1,0)]
def slice_report104(rows):
    out={}
    for s in sorted({r[0] for r in rows}):
        rr=[r for r in rows if r[0]==s]; out[s]={"n":len(rr),"base":np.mean([r[1] for r in rr]),"new":np.mean([r[2] for r in rr])}
    return out
slices104=slice_report104(rows104); macro_new104=np.mean([v["new"] for v in slices104.values()])
assert slices104["normal"]["new"]>slices104["normal"]["base"]
assert slices104["high"]["new"]<slices104["high"]["base"]
assert math.isclose(float(macro_new104),.75)

## 7. 防止训练污染与评测过拟合

对输入、参考答案及规范化版本做指纹；训练/提示示例与 eval 做 exact/near-duplicate 审计。不能证明未污染时要标注结果限制。频繁手调同一测试集也会发生“人工过拟合”，因此保留不可见 holdout 和轮换题集。

In [ ]:
def fingerprint104(text): return hashlib.sha256(" ".join(normalize104(text)).encode()).hexdigest()
train104=["退款期 7 天","如何重置密码"]; eval_text104=["退款期是7天","新的独立问题"]
train_fp104={fingerprint104(x) for x in train104}; exact_overlap104=[x for x in eval_text104 if fingerprint104(x) in train_fp104]
shingles104=lambda s:set(zip(normalize104(s),normalize104(s)[1:]))
jac104=lambda a,b:len(a&b)/len(a|b) if a|b else 1.
near104=max(jac104(shingles104(train104[0]),shingles104(eval_text104[0])),0)
assert exact_overlap104==[]
assert 0<=near104<=1 and fingerprint104("A B")==fingerprint104("a b")
assert fingerprint104(train104[0])!=fingerprint104(eval_text104[0])

## 8. 离线门禁、线上验证与失败闭环

发布规则应机器可读：总体质量最低增益、高风险零回归、安全率、p95 与成本上限。通过离线门禁只允许进入 shadow/canary；线上监控分布、拒答、工具错误与人工升级。新失败去重标注后进入下一版 challenge set。

In [ ]:
metrics104={"quality_gain":.08,"high_risk_regressions":0,"unsafe_rate":.002,"p95":1.1,"cost":.014}; gates104={"quality_gain":.03,"high_risk_regressions":0,"unsafe_rate":.005,"p95":1.2,"cost":.02}
passed104=metrics104["quality_gain"]>=gates104["quality_gain"] and metrics104["high_risk_regressions"]<=gates104["high_risk_regressions"] and all(metrics104[k]<=gates104[k] for k in ("unsafe_rate","p95","cost"))
manifest104={"schema":1,"suite":"agent-gold-v5","cases":len(cases104),"graders":["exact","trace","human"],"gates":gates104}; digest104=hashlib.sha256(json.dumps(manifest104,sort_keys=True).encode()).hexdigest()
assert passed104
assert manifest104["cases"]==4 and "trace" in manifest104["graders"]
assert len(digest104)==64

## 面试总结

一套成熟评测回答包含：**任务合同与 slice → 多来源黄金集 → 确定性 grader 优先 → Agent 轨迹 → paired 置信区间 → 高风险门禁 → 污染审计 → shadow/canary 与失败回流**。评测本身也是软件和数据产品，必须版本化、单测和监控。

延伸阅读：[HELM](https://arxiv.org/abs/2211.09110)、[NIST 生成式 AI 风险画像](https://www.nist.gov/publications/artificial-intelligence-risk-management-framework-generative-artificial-intelligence)、[Agent Evals 实践](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents)。